<a href="https://colab.research.google.com/github/qingsiruy/my_Modern-Computer-Vision-with-PyTorch/blob/main/Chapter08/Training_SSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
# 检查是否存在存放数据的文件夹，避免重复下载
if not os.path.exists('open-images-bus-trucks'):
    # 安装必要的工具库 torch_snippets
    %pip install -q torch_snippets
    # 从云端下载公交车和卡车的图像数据集压缩包
    !wget --quiet https://www.dropbox.com/s/agmzwk95v96ihic/open-images-bus-trucks.tar.xz
    # 解压下载好的数据集
    !tar -xf open-images-bus-trucks.tar.xz
    # 删除压缩包以节省空间
    !rm open-images-bus-trucks.tar.xz
    # 从 GitHub 克隆目标检测（SSD）相关的实用工具代码库
    !git clone https://github.com/sizhky/ssd-utils/
# 进入下载好的工具库目录
%cd ssd-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.3/110.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.2/398.2 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.5/470.5 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.3 MB/s eta 0:00:00
Cl

In [2]:
from torch_snippets import *
import glob
# 设置数据集的根目录和图片目录
DATA_ROOT = '../open-images-bus-trucks/'
IMAGE_ROOT = f'{DATA_ROOT}/images'
# 读取包含图片标注信息（如物体位置和标签）的 CSV 文件
DF_RAW = df = pd.read_csv(f'{DATA_ROOT}/df.csv')

# 过滤数据，确保只包含唯一的图片 ID
df = df[df['ImageID'].isin(df['ImageID'].unique().tolist())]

# 将类别名称（如 'Bus'）映射为数字 ID，方便计算机处理
label2target = {l:t+1 for t,l in enumerate(DF_RAW['LabelName'].unique())}
# 添加一个背景类别，ID 设为 0
label2target['background'] = 0
# 创建一个反向映射：从数字 ID 找回类别名称
target2label = {t:l for l,t in label2target.items()}
background_class = label2target['background']
num_classes = len(label2target)

# 自动选择运行设备：如果有显卡则用显卡 (cuda)，否则用 CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
import collections, os, torch
from PIL import Image
from torchvision import transforms

# 图像标准化：让图片的像素值符合模型预期的分布（加速训练）
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
# 反标准化：用于将处理后的数据变回可见图片（虽然这里定义了但在预处理中没直接用）
denormalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.255],
    std=[1/0.229, 1/0.224, 1/0.255]
)

# 图像预处理函数：将图片转为 PyTorch 张量并调整通道顺序及标准化
def preprocess_image(img):
    img = torch.tensor(img).permute(2,0,1)
    img = normalize(img)
    return img.to(device).float()

# 定义数据集加载类，负责把硬盘里的图片读进内存并整理好
class OpenDataset(torch.utils.data.Dataset):
    w, h = 300, 300 # SSD 模型要求的输入尺寸通常是 300x300
    def __init__(self, df, image_dir=IMAGE_ROOT):
        self.image_dir = image_dir
        self.files = glob.glob(self.image_dir+'/*')
        self.df = df
        self.image_infos = df.ImageID.unique()
        logger.info(f'{len(self)} items loaded')

    def __getitem__(self, ix):
        # 根据索引读取一张图片及其对应的标注框和标签
        image_id = self.image_infos[ix]
        img_path = find(image_id, self.files)
        img = Image.open(img_path).convert("RGB")
        # 缩放图片到 300x300 并归一化到 0-1 之间
        img = np.array(img.resize((self.w, self.h), resample=Image.BILINEAR))/255.
        data = df[df['ImageID'] == image_id]
        labels = data['LabelName'].values.tolist()
        # 获取物体框的坐标 (左上, 右下)
        data = data[['XMin','YMin','XMax','YMax']].values
        # 将相对比例坐标转换为 300x300 的绝对坐标
        data[:,[0,2]] *= self.w
        data[:,[1,3]] *= self.h
        boxes = data.astype(np.uint32).tolist()
        return img, boxes, labels

    def collate_fn(self, batch):
        # 整理函数：把多个图片打包成一个批次 (Batch) 发送给模型
        images, boxes, labels = [], [], []
        for item in batch:
            img, image_boxes, image_labels = item
            img = preprocess_image(img)[None] # 添加批次维度
            images.append(img)
            # 将坐标再次转为 0-1 之间的比例，方便模型训练
            boxes.append(torch.tensor(image_boxes).float().to(device)/300.)
            labels.append(torch.tensor([label2target[c] for c in image_labels]).long().to(device))
        images = torch.cat(images).to(device)
        return images, boxes, labels

    def __len__(self):
        return len(self.image_infos)

In [4]:
from sklearn.model_selection import train_test_split
# 将数据集随机切分为 90% 训练集和 10% 验证集
trn_ids, val_ids = train_test_split(df.ImageID.unique(), test_size=0.1, random_state=99)
trn_df, val_df = df[df['ImageID'].isin(trn_ids)], df[df['ImageID'].isin(val_ids)]

# 初始化数据集对象
train_ds = OpenDataset(trn_df)
test_ds = OpenDataset(val_df)

# 创建 DataLoader：用于在训练时按批次 (每批4张) 自动喂数据给模型
train_loader = DataLoader(train_ds, batch_size=4, collate_fn=train_ds.collate_fn, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=4, collate_fn=test_ds.collate_fn, drop_last=True)

[06/11/26 17:19:56] INFO     13702 items loaded                                                                                                    ]8;id=315578;file:///tmp/ipykernel_5585/1904448957.py:25\1904448957.py]8;;\:]8;id=258559;file:///tmp/ipykernel_5585/1904448957.py:25#__init__:25\__init__:25]8;;\

                    INFO     1523 items loaded                                                                                                     ]8;id=17400;file:///tmp/ipykernel_5585/1904448957.py:25\1904448957.py]8;;\:]8;id=770571;file:///tmp/ipykernel_5585/1904448957.py:25#__init__:25\__init__:25]8;;\

In [5]:
def train_batch(inputs, model, criterion, optimizer):
    # 训练模式：计算梯度并更新模型参数
    model.train()
    images, boxes, labels = inputs
    # 模型预测
    _regr, _clss = model(images)
    # 计算损失（预测值与真实值的差距）
    loss = criterion(_regr, _clss, boxes, labels)
    # 反向传播和优化步：这一步是模型“学习”的关键
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss

@torch.no_grad() # 验证时不需要计算梯度，节省内存和时间
def validate_batch(inputs, model, criterion):
    # 评估模式：只看预测效果，不更新参数
    model.eval()
    images, boxes, labels = inputs
    _regr, _clss = model(images)
    loss = criterion(_regr, _clss, boxes, labels)
    return loss

In [6]:
# 从本地工具库导入 SSD 模型架构和多框损失函数
from model import SSD300, MultiBoxLoss
from detect import *

In [7]:
n_epochs = 3 # 训练的总轮数（将数据完整过几遍）

# 初始化 SSD300 模型
model = SSD300(num_classes, device)
# 设置优化器：AdamW 是一种常用的自动调整学习率的算法
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
# 定义损失函数：用于衡量位置偏差和类别识别误差
criterion = MultiBoxLoss(priors_cxcy=model.priors_cxcy, device=device)

# 记录器：用于在界面上实时显示训练进度和损失曲线
log = Report(n_epochs=n_epochs)
logs_to_print = 5

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 124MB/s]



Loaded base model.



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


In [ ]:
# 核心训练循环
for epoch in range(n_epochs):
    _n = len(train_loader)
    for ix, inputs in enumerate(train_loader):
        # 进行一个批次的训练
        loss = train_batch(inputs, model, criterion, optimizer)
        # 记录当前轮次的训练损失
        pos = (epoch + (ix+1)/_n)
        log.record(pos, trn_loss=loss.item(), end='\r')

    _n = len(test_loader)
    for ix,inputs in enumerate(test_loader):
        # 进行一个批次的验证
        loss = validate_batch(inputs, model, criterion)
        # 记录验证损失
        pos = (epoch + (ix+1)/_n)
        log.record(pos, val_loss=loss.item(), end='\r')

In [ ]:
image_paths = Glob(f'{DATA_ROOT}/images/*')
image_id = choose(test_ds.image_infos)
img_path = find(image_id, test_ds.files)
original_image = Image.open(img_path, mode='r')
original_image = original_image.convert('RGB')

In [ ]:
# 测试环节：随机选几张图看看模型预测得准不准
image_paths = Glob(f'{DATA_ROOT}/images/*')
for _ in range(3):
    # 随机选一张测试集的图
    image_id = choose(test_ds.image_infos)
    img_path = find(image_id, test_ds.files)
    original_image = Image.open(img_path, mode='r').convert('RGB')
    # 使用训练好的模型进行检测：得到物体框 (bbs)、标签 (labels) 和置信度 (scores)
    bbs, labels, scores = detect(original_image, model, min_score=0.9, max_overlap=0.5, top_k=200, device=device)

    # 将结果转换为可读文本
    labels = [target2label[c.item()] for c in labels]
    label_with_conf = [f'{l} @ {s:.2f}' for l,s in zip(labels,scores)]

    # 打印结果并画图显示，bbs 是物体框的位置，texts 是标签和分数
    print(bbs, label_with_conf)
    show(original_image, bbs=bbs, texts=label_with_conf, text_sz=10)